In [2]:
# Importing Libraries to start
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Testing to make sure it all imported right")

Testing to make sure it all imported right


In [3]:
# Fetching the calendar page
url = "https://www.ccny.cuny.edu/registrar/fall"

r = requests.get(url)

# Checking status, the 200 are good 
print(r.status_code)
print(r.text[:500])  # printing first 500 characters of the HTML

200
<!DOCTYPE html>
<html lang="en" dir="ltr" prefix="content: http://purl.org/rss/1.0/modules/content/  dc: http://purl.org/dc/terms/  foaf: http://xmlns.com/foaf/0.1/  og: http://ogp.me/ns#  rdfs: http://www.w3.org/2000/01/rdf-schema#  schema: http://schema.org/  sioc: http://rdfs.org/sioc/ns#  sioct: http://rdfs.org/sioc/types#  skos: http://www.w3.org/2004/02/skos/core#  xsd: http://www.w3.org/2001/XMLSchema# ">
  <head>
    <meta charset="utf-8" />
<meta name="icbm" content="40.8200471,-73.9492


In [4]:
# Parsing the HMTL with Beautiful Soup and getting the table
soup = BeautifulSoup(r.text, 'html.parser')

table = soup.find('table', {'border': '3'})
print(table)

<table align="center" border="3" cellpadding="1" cellspacing="1" style="width:1272px">
<thead>
<tr>
<th class="text-align-left" scope="col" style="width: 305px;">DATES</th>
<th class="text-align-left" scope="col" style="width: 327px;">DAYS</th>
<th scope="col" style="width: 617px;"> </th>
</tr>
</thead>
<tbody>
<tr>
<td style="width:305px">
<p><strong>August 01</strong></p>
</td>
<td style="width:327px">
<p><strong>Sunday</strong></p>
</td>
<td style="width:617px">
<p><strong>Application for degree for January and February 2022 begins</strong></p>
</td>
</tr>
<tr>
<td style="width:305px">
<p>August 18</p>
</td>
<td style="width:327px">
<p>Wednesday</p>
</td>
<td style="width:617px">
<p>Last day to apply for Study Abroad</p>
</td>
</tr>
<tr>
<td style="width:305px">
<p><strong>August 24</strong></p>
</td>
<td style="width:327px">
<p><strong>Tuesday</strong></p>
</td>
<td style="width:617px">
<p><strong>Last day of Registration;<br/>
			Last day to file ePermit for the Fall 2021;<br/>
		

In [5]:
# extracting rows from the table
rows = []

for tr in table.find_all('tr'):
    cells = tr.find_all('td')
    if len(cells) == 3:  # skipping rows that don't have 3 cols 
        date = cells[0].get_text(strip=True)
        dow = cells[1].get_text(strip=True)
        text = cells[2].get_text(strip=True)
        rows.append({'date': date, 'dow': dow, 'text': text})

print(rows[:5])  # print first couple rows to check

[{'date': 'August 01', 'dow': 'Sunday', 'text': 'Application for degree for January and February 2022 begins'}, {'date': 'August 18', 'dow': 'Wednesday', 'text': 'Last day to apply for Study Abroad'}, {'date': 'August 24', 'dow': 'Tuesday', 'text': 'Last day of Registration;Last day to file ePermit for the Fall 2021;Last day to drop classes for 100% tuition refund;'}, {'date': 'August 25', 'dow': 'Wednesday', 'text': 'Start of Fall Term;Classes begin;Initial Registration Appeals begin;'}, {'date': 'August 25 - 31', 'dow': 'Wednesday - Tuesday', 'text': 'Change of program period; late fees apply'}]


In [ ]:
# Building the DataFrame with a date index
from datetime import datetime

df = pd.DataFrame(rows)

# Convert date strings to actual date objects (taking first date if range)
def parse_date(date_str):
    date_str = date_str.split('-')[0].strip()  # take first date if range
    
    # check if date already has a year (4 digit number in string) since I noticed different formats when testing
    import re
    if re.search(r'\d{4}', date_str):
        date_str = date_str.replace(',', '')  # remove comma if present
        return datetime.strptime(date_str.strip(), '%B %d %Y').date()
    else:
        date_str = date_str.replace(',', '')  # remove comma just in case
        date_str = date_str + ' 2021'
        return datetime.strptime(date_str.strip(), '%B %d %Y').date()

df['date'] = df['date'].apply(parse_date)
df = df.set_index('date')

print(df.head())

                            dow  \
date                              
2021-08-01               Sunday   
2021-08-18            Wednesday   
2021-08-24              Tuesday   
2021-08-25            Wednesday   
2021-08-25  Wednesday - Tuesday   

                                                         text  
date                                                           
2021-08-01  Application for degree for January and Februar...  
2021-08-18                 Last day to apply for Study Abroad  
2021-08-24  Last day of Registration;Last day to file ePer...  
2021-08-25  Start of Fall Term;Classes begin;Initial Regis...  
2021-08-25          Change of program period; late fees apply  


In [ ]:
# Displaying the final DataFrame nicely
pd.set_option('display.max_colwidth', None)  
print(df.to_string())

                             dow                                                                                                                                                                                                                                                                                text
date                                                                                                                                                                                                                                                                                                                
2021-08-01                Sunday                                                                                                                                                                                                                         Application for degree for January and February 2022 begins
2021-08-18             Wednesday                                         